In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 15. Multi-Head Attention — Viewing from Multiple Perspectives

> **Learning Objectives**
> - Understand why Multi-Head Attention uses multiple heads in parallel
> - Explain the relationship between the number of heads $h$ and head dimension $d_k = d/h$
> - Compare the differences and memory/speed effects of MHA, MQA, and GQA

## 15.1 Limitations of a Single Head

A single attention head struggles to learn more than one type of relationship (e.g., subject-verb). How can we **learn multiple relationships simultaneously**?

## 15.2 Multi-Head Attention (MHA)

$$\mathrm{MultiHead}(Q, K, V) = \mathrm{Concat}(\mathrm{head}_1, \ldots, \mathrm{head}_h) W^O$$
$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W_i^K, V W_i^V)$$

- $W_i^Q \in \mathbb{R}^{d \times d_k}$, $W_i^K \in \mathbb{R}^{d \times d_k}$, $W_i^V \in \mathbb{R}^{d \times d_v}$
- $W^O \in \mathbb{R}^{h d_v \times d}$
- Typically $d_k = d_v = d/h$

Example: $d = 512, h = 8$ → $d_k = 64$. Each head operates in a 64-dimensional space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # Q, K, V, O projections (in one pass)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x, mask=None):
        B, T, D = x.shape
        # (B, T, 3D) -> three (B, T, D) tensors
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        # gradients Dimension text: (B, T, D) -> (B, T, H, d_k) -> (B, H, T, d_k)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        # Attention: (B, H, T, d_k) @ (B, H, d_k, T) -> (B, H, T, T)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        # (B, H, T, d_k) -> (B, T, H, d_k) -> (B, T, D)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# text
d_model, n_heads = 64, 8
mha = MultiHeadAttention(d_model, n_heads)
x = torch.randn(2, 10, d_model)
out, weights = mha(x)
print(f"Input: {x.shape}")
print(f"Output: {out.shape}")
print(f"Attention Weight: {weights.shape} (B, H, T, T)")
print(f"Parameter Count: {sum(p.numel() for p in mha.parameters()):,}")


## 15.3 Head Interpretability

Different heads tend to learn different relationships:
- Some heads: reference to previous tokens (positional head)
- Some heads: syntactic relationships (subject-verb)
- Some heads: semantic similarity

In [ ]:
# text Attention text Visualization
torch.manual_seed(0)
mha = MultiHeadAttention(d_model=32, n_heads=4)

# Simple sequence (4 tokens)
x = torch.randn(1, 8, 32)
out, weights = mha(x)

# Attention pattern for each head
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
tokens = [f'tok{i}' for i in range(8)]
for h, ax in enumerate(axes):
    im = ax.imshow(weights[0, h].detach().numpy(), cmap='Blues', vmin=0, vmax=1)
    ax.set_title(f'Head {h}')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch15_heads_patterns.png', dpi=100, bbox_inches='tight')
plt.show()
print("Each head learns a different attention pattern (learning diverse relationships).")


## 15.4 Multi-Query Attention (MQA)

**Problem**: MHA requires KV cache of size $O(L \cdot h \cdot d_k)$, leading to large memory usage during inference.

**MQA** (Shazeer, 2019): Only $Q$ is projected per head; $K$ and $V$ are shared across all heads.

$$\mathrm{head}_i = \mathrm{Attention}(Q W_i^Q, K W^K, V W^V)$$

- Parameter savings, KV cache reduced by factor of $1/h$
- Slight quality drop, but significant inference speedup

## 15.5 Grouped-Query Attention (GQA)

**GQA** (Ainslie et al., 2023): A middle ground.
- $h$ query heads, $g$ key-value head groups ($1 \leq g \leq h$)
- $g=1$: MQA, $g=h$: MHA
- **Choice in LLaMA-2** (typically $g = h/8$)

In [ ]:
# MHA, MQA, GQA text Comparison
class GroupedQueryAttention(nn.Module):
    def __init__(self, d_model, n_q_heads, n_kv_heads):
        super().__init__()
        assert d_model % n_q_heads == 0
        self.d_model = d_model
        self.n_q_heads = n_q_heads
        self.n_kv_heads = n_kv_heads
        self.d_k = d_model // n_q_heads
        # Q: n_q_heads * d_k, K/V: n_kv_heads * d_k
        self.W_q = nn.Linear(d_model, n_q_heads * self.d_k, bias=False)
        self.W_k = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_v = nn.Linear(d_model, n_kv_heads * self.d_k, bias=False)
        self.W_o = nn.Linear(n_q_heads * self.d_k, d_model, bias=False)
        # Number of Q heads per group
        self.n_rep = n_q_heads // n_kv_heads

    def forward(self, x, mask=None):
        B, T, D = x.shape
        q = self.W_q(x).view(B, T, self.n_q_heads, self.d_k).transpose(1, 2)
        k = self.W_k(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        v = self.W_v(x).view(B, T, self.n_kv_heads, self.d_k).transpose(1, 2)
        # Repeat KV to match the number of Q heads (group expansion)
        if self.n_rep > 1:
            k = k.repeat_interleave(self.n_rep, dim=1)
            v = v.repeat_interleave(self.n_rep, dim=1)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, -1)
        return self.W_o(out), weights

# Comparison: MHA, GQA, MQA
d = 64
configs = [
    ('MHA (h=8)', 8, 8),
    ('GQA (g=4)', 8, 4),
    ('GQA (g=2)', 8, 2),
    ('MQA (g=1)', 8, 1),
]
print(f"{'Config':>12} | {'Params':>10} | {'KV size':>10}")
print("-" * 38)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(d, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    kv_size = nkv * (d // nq)  # dimension per KV head
    print(f"{name:>12} | {n_params:>10,} | {kv_size:>10}")


## 15.6 [CPU/GPU Benchmark ⑥] MHA vs MQA vs GQA

Comparison of KV cache memory and speed during inference.

In [ ]:
# MHA/GQA/MQA Inference Benchmark
from llm_math.bench import time_fn

configs = [
    ('MHA', 8, 8),
    ('GQA-4', 8, 4),
    ('GQA-2', 8, 2),
    ('MQA', 8, 1),
]
B, T, D = 1, 256, 256
x = torch.randn(B, T, D)

print(f"\n{'Config':>8} | {'Params':>8} | {'CPU (ms)':>12} | {'GPU (ms)':>12}")
print("-" * 50)
for name, nq, nkv in configs:
    attn = GroupedQueryAttention(D, nq, nkv)
    n_params = sum(p.numel() for p in attn.parameters())
    res = time_fn(lambda x: attn(x), x, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        attn_gpu = GroupedQueryAttention(D, nq, nkv).cuda()
        x_gpu = x.cuda()
        res_gpu = time_fn(lambda x: attn_gpu(x), x_gpu, device='cuda', warmup=2, repeat=5)
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f}")
    else:
        print(f"{name:>8} | {n_params:>8,} | {res['mean_ms']:>12.3f} | {'N/A':>12}")


## 15.7 Key Takeaways

| Variant | Q Heads | KV Heads | KV Cache | Use Case |
|---|---|---|---|---|
| MHA | $h$ | $h$ | $O(Lhd_k)$ | Quality First |
| MQA | $h$ | 1 | $O(Ld_k)$ | Speed First |
| GQA | $h$ | $g$ | $O(Lgd_k)$ | LLaMA-2 Standard |

## Exercises

1. In MHA, vary the number of heads among 1, 4, 8, and 16, and compare learning curves.
2. Visualize attention head patterns and check whether a "positional head" (focused on the diagonal) exists.
3. Calculate the difference in parameter count between MQA and MHA, and verify experimentally.
4. In GQA, compare inference speed and memory usage for $g = 1, 2, 4, 8$.
5. Explain why the $W^O$ matrix in Multi-Head Attention is necessary.

> Solution: `solutions/ch15_solutions.ipynb`